# Ejecuta e inspecciona la búsqueda por patrones

Nivel: 2. Requisitos previos: Nivel 1 de este tema y niveles 1-2 de conminución. Tiempo de lectura: 30 minutos.

## Objetivos

Ejecuta el optimizador de producción OreFlow para el pórfido blando, inspecciona su traza real de evaluaciones y comprueba si el punto final satisface cada restricción física declarada.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Configura las importaciones y registra versiones. Este paso hace reproducible la ejecución del método.

In [1]:
# Importa herramientas de análisis y registra sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resuelve el pipeline local desde la estructura relativa del repositorio.
repo = Path('../../../').resolve()
sys.path.insert(0, str(repo / 'data-pipeline'))
for package in ('numpy', 'scipy', 'pandas', 'matplotlib', 'nbclient', 'nbformat', 'ipykernel'):
    print(package, importlib.metadata.version(package))

numpy 2.2.6
scipy 1.17.1
pandas 2.2.3
matplotlib 3.11.2
nbclient 0.11.0
nbformat 5.11.1
ipykernel 7.4.0


## Datos

Responsable: autores de OreFlow; el caso de pórfido blando y el Contrato 1 están versionados en `data/derived/cases/copper_porphyry_soft.json` y `data/derived/contract/operating_contract.json`, licencia MIT del proyecto. Repositorio: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

La búsqueda generalizada por patrones prueba vecinos de malla en coordenadas de decisión acotadas y aplica una barrera progresiva a violaciones de ley, potencia y agua. Este notebook llama `optimize` desde `pipeline/methods/optimization.py`; no reproduce el optimizador. La corrida desactiva el filtro surrogate opcional para aislar la búsqueda evaluada por el motor.

Carga el caso de producción, el contrato y el optimizador; ejecuta una búsqueda determinista de recuperación de metal sin trayectoria de ponderaciones.

In [2]:
from pipeline.cases.catalog import CASES
from pipeline.cases.catalog import CASES
from pipeline.methods.optimization import optimize

# Paso ejecutable del método OreFlow explicado en la celda anterior.
case = next(item for item in CASES if item.id == 'copper_porphyry_soft')
contract = json.loads((repo / 'data/derived/contract/operating_contract.json').read_text(encoding='utf-8'))
search = optimize(case, case.nominal, contract, weight=1.0, path=False, screen=None)
print('status:', search['status'], '| engine evaluations:', search['evaluations'])
print('screened:', search['screened'], '| decision variables:', search['decisions'])
print('feasible optimum:', search['optimum']['feasible'] if search['optimum'] else False)

status: optimal | engine evaluations: 266
screened: False | decision variables: ['target_p80_um', 'collector_gpt', 'jg_cm_s']
feasible optimum: True


Inspecciona las iteraciones registradas de búsqueda por patrones e identifica la condición de parada de malla. Un presupuesto finito puede terminar antes de converger la malla.

In [3]:
# Inspecciona las iteraciones registradas de búsqueda por patrones e identifica la condición de parada de malla.
print('start count:', len(search['starts']))
display(pd.DataFrame([{key: start.get(key) for key in ('start', 'feasible', 'evaluations', 'stop')} for start in search['starts']]))
print('converged:', search.get('converged'), '| feasible:', search['optimum']['feasible'])

start count: 6


,start,feasible,evaluations,stop
0,"{'target_p80_um': 150.0, 'collector_gpt': 25.0...",None,158,mesh
1,"{'target_p80_um': 131.25, 'collector_gpt': 18....",None,96,mesh
2,"{'target_p80_um': 243.75, 'collector_gpt': 56....",None,4,mesh
3,"{'target_p80_um': 131.25, 'collector_gpt': 56....",None,0,mesh
4,"{'target_p80_um': 243.75, 'collector_gpt': 18....",None,2,mesh
5,"{'target_p80_um': 187.5, 'collector_gpt': 37.5...",None,6,mesh


converged: True | feasible: True


Verifica la holgura final de ley, potencia instalada y agua; luego compara base y óptimo en unidades de proceso.

In [4]:
# Verifica la holgura final de ley, potencia instalada y agua; luego compara base y óptimo en unidades de proceso.
optimum = search['optimum']
print('constraint slacks:', optimum['slacks'])
comparison = pd.DataFrame([
    {'state': 'base', 'recovered primary (t/h)': search['base']['recovered_tph'], **search['base']['values']},
    {'state': 'optimum', 'recovered primary (t/h)': optimum['recovered_tph'], **optimum['values']},
]).set_index('state')
display(comparison)

constraint slacks: {'grade': 1.0715590708919507, 'power': 0.6730429101571644, 'water': 0.07950103059474811}


,recovered primary (t/h),grade,required_power_kw,water_m3_t,energy_kwh_t
state,,,,,
base,4.913088,26.415673,6456.494529,2.174074,10.394629
optimum,5.195311,25.071559,7269.326957,2.200499,11.868042


## Resultados

La corrida ejecuta la función de producción usada por el bake. Las holguras indican la distancia a cada límite. Si la búsqueda termina por presupuesto, entrega el mejor punto factible encontrado; no demuestra estacionariedad ni optimalidad global.

## Resumen

Ejecutaste el optimizador y verificaste su traza y holguras. Que la búsqueda sea reproducible no convierte el resultado de escenario en un óptimo de planta.

## Referencias

- https://doi.org/10.1137/S1052623493250780
- https://doi.org/10.1137/070692662